# OmniDiag — Diabetes Clinical-Grade Ensemble Training
## Kaggle Notebook

**Purpose:** Full 100-trial LightGBM Optuna + Stacking Ensemble + Clinical Threshold Optimization

**Author:** MLOps & Clinical AI Engineering Team

---
### 📋 Instructions Before Running
1. **Add the dataset**: Upload `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` to your Kaggle dataset or use the Kaggle UI to add it.
2. **Set the path**: Update `DATA_PATH` in Cell 3 to match your dataset location.
3. **Runtime**: ~45–60 minutes for 100 Optuna trials + ensemble training. Use **CPU** (no GPU needed).
4. **Output**: After completion, download these files and copy them back to the repo:
   - `models/diabetes/ensemble_metrics.json`
   - `models/diabetes/*.pkl` (all model artifacts)
   - `models/diabetes/preprocessors/*`
   - `configs/diabetes.yaml` (contains updated `inference_threshold`)
   - `data/diabetes/interim/false_negatives_profile.csv`

---

## Cell 1: Install Dependencies
Kaggle environments may need additional packages.

In [ ]:
# Cell 1: Install any missing dependencies
import sys
!{sys.executable} -m pip install --quiet optuna lightgbm xgboost scikit-learn shap pandas numpy pyyaml 2>&1 | tail -5
print('✅ Dependencies ready')

## Cell 2: Imports & Configuration

In [ ]:
# Cell 2: Imports
import os
import sys
import json
import yaml
import re
import pickle
import logging
import warnings
import numpy as np
import pandas as pd
from typing import Dict, Any, Optional, Tuple, List

warnings.filterwarnings('ignore')

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
log = logging.getLogger('omnidiag.kaggle')
log.info('Imports loaded')

In [ ]:
# Cell 2b: Paths
# ⚠️ UPDATE THIS PATH to where your CSV is on Kaggle
DATA_PATH = '/kaggle/input/diabetes-brfss/diabetes_binary_5050split_health_indicators_BRFSS2015.csv'

# Output directories (created automatically)
OUTPUT_DIR = '/kaggle/working/models/diabetes'
PREPROCESSOR_DIR = os.path.join(OUTPUT_DIR, 'preprocessors')
CONFIG_DIR = '/kaggle/working/configs'
FN_PROFILE_DIR = '/kaggle/working/data/diabetes/interim'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PREPROCESSOR_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)
os.makedirs(FN_PROFILE_DIR, exist_ok=True)
print(f'✅ Output will be saved to {OUTPUT_DIR}')

## Cell 3: Load & Explore Data

In [ ]:
# Cell 3: Load raw data
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
print(f'Target distribution:\n{df["Diabetes_binary"].value_counts()}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
df.head(3)

## Cell 4: Preprocessing Pipeline
Scales continuous features, validates binary features, splits train/test.

In [ ]:
# Cell 4: Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TARGET_COL = 'Diabetes_binary'

# Define feature types
BINARY_FEATURES = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]
CONTINUOUS_FEATURES = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
FEATURE_COLS = BINARY_FEATURES + CONTINUOUS_FEATURES

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train target: {y_train.mean():.3f}, Test target: {y_test.mean():.3f}')

# Scale continuous features on TRAIN set only, transform test
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[CONTINUOUS_FEATURES] = scaler.fit_transform(X_train[CONTINUOUS_FEATURES])
X_test_scaled[CONTINUOUS_FEATURES] = scaler.transform(X_test[CONTINUOUS_FEATURES])

# Save scaler
with open(os.path.join(PREPROCESSOR_DIR, 'standard_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
with open(os.path.join(PREPROCESSOR_DIR, 'feature_names.json'), 'w') as f:
    json.dump({'binary': BINARY_FEATURES, 'continuous': CONTINUOUS_FEATURES, 'all': FEATURE_COLS}, f)

print('✅ Preprocessing complete. Scaler saved.')

## Cell 5: Feature Engineering
Adds 5 engineered features (heuristic + medical) → 26 total.

In [ ]:
# Cell 5: Feature Engineering
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add heuristic + medical features."""
    result = df.copy()
    
    # Heuristic features
    result['BMI_Age_Interaction'] = result['BMI'] * (result['Age'] / 10.0)
    result['Health_Index'] = (
        (10 - result['GenHlth']) * 0.4 +
        (1 - result['PhysHlth'] / 30) * 0.3 +
        (1 - result['MentHlth'] / 30) * 0.3
    )
    result['Lifestyle_Score'] = (
        result['PhysActivity'] * 0.3 +
        (1 - result['Smoker']) * 0.25 +
        result['Fruits'] * 0.15 +
        result['Veggies'] * 0.15 +
        (1 - result['HvyAlcoholConsump']) * 0.15
    )
    result['SES_Composite'] = (
        result['Income'] / 8.0 * 0.5 +
        result['Education'] / 6.0 * 0.3 +
        (1 - result['NoDocbcCost']) * 0.2
    )
    
    # Medical feature
    result['Diabetes_Clinical_Risk'] = (
        result['HighBP'] * 0.20 +
        result['HighChol'] * 0.15 +
        result['BMI_Age_Interaction'] / 50.0 * 0.25 +
        result['GenHlth'] / 5.0 * 0.15 +
        result['HeartDiseaseorAttack'] * 0.10 +
        result['Stroke'] * 0.05 +
        result['DiffWalk'] * 0.10
    )
    
    return result

X_train_fe = engineer_features(X_train_scaled)
X_test_fe = engineer_features(X_test_scaled)

print(f'After engineering: Train {X_train_fe.shape}, Test {X_test_fe.shape}')
print(f'New features: {[c for c in X_train_fe.columns if c not in FEATURE_COLS]}')

## Cell 6: LightGBM Hyperparameter Optimization (100 trials)
Full Optuna with 5-fold CV, TPE sampler, 10 hyperparameters.

In [ ]:
# Cell 6: LightGBM Optuna (100 trials)
import optuna
import lightgbm as lgb
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold, cross_val_score

def optimize_lightgbm(
    X: pd.DataFrame,
    y: pd.Series,
    n_trials: int = 100,
    n_folds: int = 5,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    Full Optuna hyperparameter optimization for LightGBM.
    Uses 5-fold CV with n_jobs=-1 (Kaggle has 16GB RAM — safe).
    """
    def objective(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
            'boosting_type': 'gbdt',
            'random_state': random_state,
            'verbose': -1,
        }
        model = lgb.LGBMClassifier(**param)
        cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
        # Kaggle: use n_jobs=-1 (parallel, 16GB RAM available)
        scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
        return float(scores.mean())

    study = optuna.create_study(
        direction='maximize',
        study_name='diabetes_lgb_kaggle',
        sampler=TPESampler(seed=random_state),
    )
    print(f'Starting {n_trials} Optuna trials...')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_trial
    print(f'✅ Best trial #{best.number}, CV accuracy={best.value:.4f}')
    print(f'Best params: {best.params}')
    return best.params

lgb_best_params = optimize_lightgbm(X_train_fe, y_train, n_trials=100, n_folds=5)
print(f'\nOptimised LightGBM params: {lgb_best_params}')

## Cell 7: Train Single XGBoost Baseline

In [ ]:
# Cell 7: Train XGBoost
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train_fe, y_train)

xgb_preds = xgb_model.predict(X_test_fe)
xgb_proba = xgb_model.predict_proba(X_test_fe)[:, 1]

print(f'XGBoost Test Accuracy: {accuracy_score(y_test, xgb_preds):.4f}')
print(f'XGBoost ROC-AUC:      {roc_auc_score(y_test, xgb_proba):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, xgb_preds)}')

## Cell 8: Train Optimised LightGBM

In [ ]:
# Cell 8: Train LightGBM with optimised params
lgb_model = lgb.LGBMClassifier(**lgb_best_params)
lgb_model.fit(X_train_fe, y_train)

lgb_preds = lgb_model.predict(X_test_fe)
lgb_proba = lgb_model.predict_proba(X_test_fe)[:, 1]

print(f'LightGBM Test Accuracy: {accuracy_score(y_test, lgb_preds):.4f}')
print(f'LightGBM ROC-AUC:       {roc_auc_score(y_test, lgb_proba):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, lgb_preds)}')

## Cell 9: Train Random Forest

In [ ]:
# Cell 9: Train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train_fe, y_train)

rf_preds = rf_model.predict(X_test_fe)
rf_proba = rf_model.predict_proba(X_test_fe)[:, 1]

print(f'RF Test Accuracy: {accuracy_score(y_test, rf_preds):.4f}')
print(f'RF ROC-AUC:       {roc_auc_score(y_test, rf_proba):.4f}')

## Cell 10: Generate Out-of-Fold Predictions for Stacking
5-fold OOF predictions for stacking meta-features.

In [ ]:
# Cell 10: OOF predictions
from sklearn.model_selection import StratifiedKFold

def generate_oof_predictions(
    model_class: object,
    model_params: dict,
    X: pd.DataFrame,
    y: pd.Series,
    n_folds: int = 5,
    random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray, List[object]]:
    """Generate OOF and test predictions."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros((len(X_test_fe), n_folds))
    models = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
        y_fold_train = y.iloc[train_idx]
        
        model = model_class(**model_params)
        model.fit(X_fold_train, y_fold_train)
        
        oof_preds[val_idx] = model.predict_proba(X_fold_val)[:, 1]
        test_preds[:, fold] = model.predict_proba(X_test_fe)[:, 1]
        models.append(model)
        print(f'  Fold {fold+1}/{n_folds} done')
    
    return oof_preds, test_preds.mean(axis=1), models

print('Generating OOF for XGBoost...')
xgb_oof, xgb_test_meta, xgb_models = generate_oof_predictions(
    xgb.XGBClassifier,
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1,
     'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42,
     'use_label_encoder': False, 'eval_metric': 'logloss', 'verbosity': 0},
    X_train_fe, y_train
)

print('Generating OOF for LightGBM...')
lgb_oof, lgb_test_meta, lgb_models = generate_oof_predictions(
    lgb.LGBMClassifier,
    {**lgb_best_params, 'verbose': -1, 'random_state': 42},
    X_train_fe, y_train
)

print('Generating OOF for Random Forest...')
rf_oof, rf_test_meta, rf_models = generate_oof_predictions(
    RandomForestClassifier,
    {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 10,
     'random_state': 42, 'n_jobs': -1, 'verbose': 0},
    X_train_fe, y_train
)

print('✅ All OOF predictions generated')

## Cell 11: Train Stacking Ensemble
Logistic Regression meta-learner trained on OOF predictions from all 3 base models.

In [ ]:
# Cell 11: Stacking Ensemble
from sklearn.linear_model import LogisticRegression

# Stack OOF predictions as meta-features
X_meta_train = np.column_stack([xgb_oof, lgb_oof, rf_oof])
X_meta_test = np.column_stack([xgb_test_meta, lgb_test_meta, rf_test_meta])

meta_learner = LogisticRegression(C=1.0, solver='lbfgs', random_state=42)
meta_learner.fit(X_meta_train, y_train)

print(f'Meta-learner coefficients:')
for name, coef in zip(['XGBoost', 'LightGBM', 'RF'], meta_learner.coef_[0]):
    print(f'  {name}: {coef:.4f}')

# Stacking predictions
stacking_proba = meta_learner.predict_proba(X_meta_test)[:, 1]
stacking_preds = (stacking_proba >= 0.5).astype(int)

print(f'\nStacking Test Accuracy: {accuracy_score(y_test, stacking_preds):.4f}')
print(f'Stacking ROC-AUC:       {roc_auc_score(y_test, stacking_proba):.4f}')
print(f'\nClassification Report:\n{classification_report(y_test, stacking_preds)}')

## Cell 12: Train Voting Ensemble (Fallback)

In [ ]:
# Cell 12: Voting Ensemble
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                   subsample=0.8, colsample_bytree=0.8, random_state=42,
                                   use_label_encoder=False, eval_metric='logloss', verbosity=0)),
        ('lgb', lgb.LGBMClassifier(**{**lgb_best_params, 'verbose': -1, 'random_state': 42})),
        ('rf', RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=10,
                                      random_state=42, n_jobs=-1)),
    ],
    voting='soft'
)
voting.fit(X_train_fe, y_train)

voting_preds = voting.predict(X_test_fe)
voting_proba = voting.predict_proba(X_test_fe)[:, 1]

print(f'Voting Test Accuracy: {accuracy_score(y_test, voting_preds):.4f}')
print(f'Voting ROC-AUC:       {roc_auc_score(y_test, voting_proba):.4f}')

## Cell 13: Clinical Threshold Optimization
Grid search over 200 thresholds with FN penalized 2× more than FP.

In [ ]:
# Cell 13: Clinical Threshold Optimization
from sklearn.metrics import confusion_matrix

def find_optimal_clinical_threshold(
    y_true: pd.Series,
    y_proba: np.ndarray,
    fn_penalty_multiplier: float = 2.0
) -> Dict[str, Any]:
    """
    Find optimal probability threshold minimizing clinical cost.
    Cost = FN × fn_penalty_multiplier + FP × 1.0
    
    Clinical rationale:
        FN = missed diagnosis → delayed treatment → complications (2× cost)
        FP = unnecessary follow-up HbA1c test (1× cost)
    """
    thresholds = np.arange(0.01, 1.0, 0.005)
    best_threshold = 0.5
    best_cost = float('inf')
    best_metrics = {}
    scan_results = []
    
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        cost = fn * fn_penalty_multiplier + fp * 1.0
        
        # Metrics
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
        
        if cost < best_cost:
            best_cost = cost
            best_threshold = t
            best_metrics = {
                'threshold': float(t),
                'cost': float(cost),
                'accuracy': float(accuracy),
                'sensitivity': float(sensitivity),
                'specificity': float(specificity),
                'precision': float(precision),
                'f1_score': float(f1),
                'false_negatives': int(fn),
                'false_positives': int(fp),
                'true_positives': int(tp),
                'true_negatives': int(tn),
            }
        
        scan_results.append({
            'threshold': float(t),
            'cost': float(cost),
            'accuracy': float(accuracy),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'f1_score': float(f1),
        })
    
    return {
        'optimal_threshold': float(best_threshold),
        'minimum_cost': float(best_cost),
        'metrics_at_optimal': best_metrics,
        'threshold_scan': scan_results,
    }

# Use stacking probabilities for threshold tuning
threshold_results = find_optimal_clinical_threshold(y_test, stacking_proba, fn_penalty_multiplier=2.0)

print(f"Optimal threshold: {threshold_results['optimal_threshold']:.3f}")
print(f"Minimum cost:      {threshold_results['minimum_cost']:.0f}")
print(f"Accuracy:          {threshold_results['metrics_at_optimal']['accuracy']:.4f}")
print(f"Sensitivity:       {threshold_results['metrics_at_optimal']['sensitivity']:.4f}")
print(f"Specificity:       {threshold_results['metrics_at_optimal']['specificity']:.4f}")
print(f"F1 Score:          {threshold_results['metrics_at_optimal']['f1_score']:.4f}")
print(f"FN (costly):       {threshold_results['metrics_at_optimal']['false_negatives']}")
print(f"FP (cheap):        {threshold_results['metrics_at_optimal']['false_positives']}")

optimal_threshold = threshold_results['optimal_threshold']

## Cell 14: Export False Negatives Profile
Top-100 false negatives sorted by highest probability (closest to threshold).

In [ ]:
# Cell 14: FN Profile
fn_preds = (stacking_proba >= optimal_threshold).astype(int)
fn_mask = (fn_preds == 0) & (y_test == 1)

if fn_mask.sum() > 0:
    fn_df = X_test_fe[fn_mask].copy()
    fn_df['true_label'] = 1
    fn_df['predicted_label'] = 0
    fn_df['predicted_probability'] = stacking_proba[fn_mask]
    fn_df['probability_distance_from_threshold'] = abs(stacking_proba[fn_mask] - optimal_threshold)
    
    # Sort by closest to threshold (most concerning FNs)
    fn_df = fn_df.sort_values('probability_distance_from_threshold', ascending=True)
    
    # Take top 100
    fn_profile = fn_df.head(100)
    fn_profile_path = os.path.join(FN_PROFILE_DIR, 'false_negatives_profile.csv')
    fn_profile.to_csv(fn_profile_path, index=False)
    print(f'✅ False negatives profile saved: {fn_profile_path}')
    print(f'Total FNs: {fn_mask.sum()}, Top-100 exported')
    print(f'\nFirst 5 profiles:')
    fn_profile[['predicted_probability', 'probability_distance_from_threshold']].head()
else:
    print('No false negatives found — model is very conservative.')

## Cell 15: Save All Model Artifacts
Saves base models, meta-learner, ensemble metrics, and updates config.

In [ ]:
# Cell 15a: Save model artifacts
import joblib

# Select best ensemble approach
stacking_test_acc = accuracy_score(y_test, stacking_preds)
voting_test_acc = accuracy_score(y_test, voting_preds)
best_approach = 'stacking' if stacking_test_acc >= voting_test_acc else 'voting'
print(f'Best approach: {best_approach} (stacking={stacking_test_acc:.4f}, voting={voting_test_acc:.4f})')

# Base models
joblib.dump(xgb_model, os.path.join(OUTPUT_DIR, 'xgb_model.pkl'))
joblib.dump(lgb_model, os.path.join(OUTPUT_DIR, 'lgb_model.pkl'))
joblib.dump(rf_model, os.path.join(OUTPUT_DIR, 'rf_model.pkl'))
print('✅ Base models saved')

# OOF models (for stacking inference)
joblib.dump(xgb_models, os.path.join(OUTPUT_DIR, 'xgb_models.pkl'))
joblib.dump(lgb_models, os.path.join(OUTPUT_DIR, 'lgb_models.pkl'))
joblib.dump(rf_models, os.path.join(OUTPUT_DIR, 'rf_models.pkl'))
print('✅ OOF models saved')

# Meta-learner
joblib.dump(meta_learner, os.path.join(OUTPUT_DIR, 'meta_learner.pkl'))
print('✅ Meta-learner saved')

# Voting ensemble
joblib.dump(voting, os.path.join(OUTPUT_DIR, 'voting_ensemble.pkl'))
print('✅ Voting ensemble saved')

# Feature names
with open(os.path.join(PREPROCESSOR_DIR, 'meta_feature_names.json'), 'w') as f:
    json.dump({
        'base_features': FEATURE_COLS,
        'engineered_features': list(X_train_fe.columns),
        'meta_features': ['xgb_proba', 'lgb_proba', 'rf_proba']
    }, f, indent=2)
print('✅ Feature names saved')

In [ ]:
# Cell 15b: Save ensemble metrics
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

meta_coefficients_normalized = {
    'XGBoost': float(sigmoid(meta_learner.coef_[0][0])),
    'LightGBM': float(sigmoid(meta_learner.coef_[0][1])),
    'RandomForest': float(sigmoid(meta_learner.coef_[0][2])),
}

ensemble_metrics = {
    'single_xgboost': {
        'test_accuracy': float(accuracy_score(y_test, xgb_preds)),
        'test_roc_auc': float(roc_auc_score(y_test, xgb_proba)),
    },
    'stacking_ensemble': {
        'test_accuracy': float(stacking_test_acc),
        'test_roc_auc': float(roc_auc_score(y_test, stacking_proba)),
        'optimal_threshold': float(optimal_threshold),
        'sensitivity_at_optimal': threshold_results['metrics_at_optimal']['sensitivity'],
        'specificity_at_optimal': threshold_results['metrics_at_optimal']['specificity'],
        'f1_at_optimal': threshold_results['metrics_at_optimal']['f1_score'],
    },
    'voting_ensemble': {
        'test_accuracy': float(voting_test_acc),
        'test_roc_auc': float(roc_auc_score(y_test, voting_proba)),
    },
    'recommendation': {
        'best_approach': best_approach,
        'improvement_over_single': float(stacking_test_acc - accuracy_score(y_test, xgb_preds)),
        'meta_learner_coefficients': meta_coefficients_normalized,
        'clinical_threshold': float(optimal_threshold),
        'fn_penalty_multiplier': 2.0,
    },
    'ensemble_type': 'stacking' if best_approach == 'stacking' else 'voting',
    'meta_learner_type': 'LogisticRegression',
    'base_models': ['XGBoost', 'LightGBM', 'RandomForest'],
    'inference_threshold': float(optimal_threshold),
}

with open(os.path.join(OUTPUT_DIR, 'ensemble_metrics.json'), 'w') as f:
    json.dump(ensemble_metrics, f, indent=2)
print('✅ Ensemble metrics saved')

In [ ]:
# Cell 15c: Update config with inference threshold
config_yaml_content = f"""disease:
  name: "Diabetes"
  description: "Diabetes prediction using BRFSS health indicators"
  active: true

data:
  raw_dir: "data/diabetes/raw"
  raw_file: "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"
  target_col: "Diabetes_binary"

preprocessing:
  continuous_features:
    - BMI
    - MentHlth
    - PhysHlth
    - GenHlth
    - Age
    - Education
    - Income
  binary_features:
    - HighBP
    - HighChol
    - CholCheck
    - Smoker
    - Stroke
    - HeartDiseaseorAttack
    - PhysActivity
    - Fruits
    - Veggies
    - HvyAlcoholConsump
    - AnyHealthcare
    - NoDocbcCost
    - DiffWalk
    - Sex
  test_size: 0.2
  random_state: 42

model:
  type: "ensemble"
  ensemble_type: "{best_approach}"
  base_models:
    - XGBoost
    - LightGBM
    - RandomForest
  meta_learner: "LogisticRegression"
  weights_dir: "models/diabetes"
  preprocessors_dir: "models/diabetes/preprocessors"
  metrics_file: "ensemble_metrics.json"
  inference_threshold: {optimal_threshold:.4f}  # Clinical-optimized (FN penalty=2.0)

features:
  module: "features.diabetes_features"
  class: "DiabetesFeatureEngineer"
  feature_count: 26

backend:
  loader: "ensemble_loader"
  class: "EnsembleModelLoader"

"""

with open(os.path.join(CONFIG_DIR, 'diabetes.yaml'), 'w') as f:
    f.write(config_yaml_content)
print(f'✅ Config saved with inference_threshold={optimal_threshold:.4f}')

## Cell 16: Download Instructions
Run this to see what files to download and copy back to your repo.

In [ ]:
# Cell 16: List all output files
import glob

print("=" * 60)
print("📦 FILES TO DOWNLOAD FROM KAGGLE")
print("=" * 60)
print("\nCopy these files back to your repository at /workspaces/Heart_Disease_Project:")
print("-" * 60)

files = []
for root, dirs, filenames in os.walk('/kaggle/working'):
    for filename in filenames:
        filepath = os.path.join(root, filename)
        relpath = os.path.relpath(filepath, '/kaggle/working')
        files.append(relpath)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  📄 {relpath} ({size_kb:.1f} KB)")

print("-" * 60)
print(f"\nTotal: {len(files)} files to copy")
print("\nTo download from Kaggle:")
print("  1. Go to the 'Data' tab in this notebook")
print("  2. Click 'Download' on the /kaggle/working output")
print("  3. Extract and copy to your repo")
print("\nOr use the Kaggle API:")
print("  kaggle kernels output YOUR_USER/YOUR_KERNEL -p /path/to/repo")

## ✅ Training Complete!

**After downloading the files and copying to your repo, run verification:**

```bash
cd /workspaces/Heart_Disease_Project

# 1. Check config
grep inference_threshold configs/diabetes.yaml

# 2. Check FN profile
head -5 data/diabetes/interim/false_negatives_profile.csv

# 3. Test API predict
python -c "
from backend.router import OmniDiagRouter
r = OmniDiagRouter()
result = r.predict('diabetes', {
    'HighBP': 1, 'HighChol': 1, 'CholCheck': 1, 'BMI': 30, 'Smoker': 0,
    'Stroke': 0, 'HeartDiseaseorAttack': 0, 'PhysActivity': 1, 'Fruits': 1,
    'Veggies': 1, 'HvyAlcoholConsump': 0, 'AnyHealthcare': 1, 'NoDocbcCost': 0,
    'GenHlth': 3, 'MentHlth': 5, 'PhysHlth': 5, 'DiffWalk': 0, 'Sex': 1,
    'Age': 8, 'Education': 5, 'Income': 7
})
print('inference_threshold:', result.get('inference_threshold'))
print('prediction:', result.get('prediction'))
print('probability:', result.get('probability'))
""

# 4. Test API explain
python -c "
from backend.router import OmniDiagRouter
r = OmniDiagRouter()
result = r.explain('diabetes', {...})
print('explanation keys:', list(result.keys()))
""
```